# Anonymization Tutorial

Use the AnalyzerEngine to scan raw text and extract PII entities.

Note: a good idea if this is a part of a pipeline, to use `langdetect` before.

In [11]:
import spacy
from spacy_langdetect import LanguageDetector
from spacy.language import Language

# 2. Register the language detector in the pipeline
@Language.factory("language_detector")
def create_language_detector(nlp, name):
    return LanguageDetector()

nlp = spacy.load("en_core_web_sm")          # or any model
nlp.add_pipe("language_detector", last=True)

# 3. Process some text
text = "Ceci est un texte en français."
doc = nlp(text)

# 4. The detected language code and score
print(doc._.language)
# e.g. {'language': 'fr', 'score': 0.99}

{'language': 'fr', 'score': 0.9999976393818567}


In [1]:
# analyzer_example.py
from presidio_analyzer import AnalyzerEngine, RecognizerResult

# Initialize the analyzer engine with default recognizers
analyzer = AnalyzerEngine()

# Sample text containing PII
text = (
    "Hello, my name is Alice Johnson. You can reach me at alice.j@example.com or +1-555-123-4567."
)

# Analyze for PII
results: [RecognizerResult] = analyzer.analyze(
    text=text,
    entities=["PERSON", "PHONE_NUMBER", "EMAIL_ADDRESS"],
    language='en'
)

# Display findings
for result in results:
    print(
        f"Entity: {result.entity_type}, "
        f"Score: {result.score:.2f}, "
        f"Span: ({result.start}, {result.end}), "
        f"Text: '{text[result.start:result.end]}'"
    )

Entity: EMAIL_ADDRESS, Score: 1.00, Span: (53, 72), Text: 'alice.j@example.com'
Entity: PERSON, Score: 0.85, Span: (18, 31), Text: 'Alice Johnson'


With the detections from the analyzer, use the AnonymizerEngine to mask or replace PII.

In [9]:
from presidio_anonymizer import AnonymizerEngine
from presidio_analyzer   import AnalyzerEngine
from presidio_anonymizer.entities import OperatorConfig

# 1. Initialize
analyzer   = AnalyzerEngine()
anonymizer = AnonymizerEngine()

# 2. Detect PII
text = "Alice Johnson, SSN: 123-45-6789, email alice.j@example.com"
results = analyzer.analyze(
    text=text,
    entities=["PERSON", "US_SSN", "EMAIL_ADDRESS"],
    language="en",
)

# 3. Build OperatorConfig with from_end
operators = {
    "DEFAULT": OperatorConfig(
        operator_name="mask",
        params={
            "masking_char": "*",
            "chars_to_mask": 0,      # 0 => mask entire entity
            "from_end": False        # must specify whether to mask from end
        }
    ),
    "EMAIL_ADDRESS": OperatorConfig(
        operator_name="replace",
        params={"new_value": "[REDACTED_EMAIL]"}
    ),
    "US_SSN": OperatorConfig(
        operator_name="replace",
        params={"new_value": "[REDACTED_SSN]"}
    ),
}

# 4. Anonymize
result = anonymizer.anonymize(
    text=text,
    analyzer_results=results,
    operators=operators,
)

print(result.text)

Alice Johnson, SSN: 123-45-6789, email [REDACTED_EMAIL]


Sometimes you need to detect domain-specific patterns. Here’s how to write a custom recognizer for credit card numbers:

In [6]:
# custom_recognizer.py
import re
from presidio_analyzer import PatternRecognizer, Pattern

class CreditCardRecognizer(PatternRecognizer):
    """
    Recognizes Credit Card numbers in the form ####-####-####-#### or ###########
    """
    PATTERNS = [
        Pattern(
            "Credit card pattern with hyphens",
            r"\b(?:\d{4}-){3}\d{4}\b",
            0.5
        ),
        Pattern(
            "Credit card pattern without hyphens",
            r"\b\d{16}\b",
            0.5
        ),
    ]

    def __init__(self, **kwargs):
        super().__init__(
            supported_entity   = "CREDIT_CARD",
            patterns           = self.PATTERNS,
            global_regex_flags = re.DOTALL|re.MULTILINE|re.IGNORECASE,
            **kwargs
        )

# Usage in analyzer
from presidio_analyzer import AnalyzerEngine

analyzer = AnalyzerEngine()
analyzer.registry.add_recognizer(CreditCardRecognizer())

text = "My credit card 1234-5678-9012-3456 will expire soon."
results = analyzer.analyze(text, entities=["CREDIT_CARD"], language="en")
for r in results:
    print(r)

type: CREDIT_CARD, start: 15, end: 34, score: 0.85
